# Two-Point Seismic Inverse Problem Demo

**Problem setting (Section 2.7).**  
An earthquake occurs at an unknown location $x \in \mathbb{S}^2$.  
Two sensors at $y_1, y_2 \in \mathbb{S}^2$ record signals
$$
I(x,y) = \exp\!\bigl(\beta(\langle x,y\rangle - 1)\bigr), \qquad U_i \mid x,y_i \sim \mathcal{N}(I(x,y_i),\sigma^2).
$$
Given measurements $(u_1,y_1),(u_2,y_2)$ we want to recover $x$ using **DPnP**.

The joint likelihood used as the data-driven score is
$$
f(x) = \exp\!\left(-\frac{(u_1 - I(x,y_1))^2}{2\sigma^2}\right)
       \exp\!\left(-\frac{(u_2 - I(x,y_2))^2}{2\sigma^2}\right),
$$
implemented as `get_two_point_seismic_f_fn` and plugged into `get_bel`.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from data import make_earth_dataloaders
from model import AmbientGeneratorScoreNet, DPnPScoreWrapper
from bel import get_bel
from utils import (
    normalize_torch,
    sample_vmf_s2,
    seismic_signal,
    get_seismic_f_fn,
    get_two_point_seismic_f_fn,
    extrinsic_to_latlon_deg_torch,
)
from test_earth import (
    run_two_point_earthquake_demo,
    cosine_similarity_vs_steps_two_point,
    plot_spherical_kde_mollweide,
)

device = "mps" if torch.backends.mps.is_available() else "cpu"
dtype  = torch.float32
print(f"Using device: {device}")

## 1. Load earthquake data and trained prior score

In [ ]:
# ---- data ----
batch_size = 512
seed = 0
train_loader, val_loader, test_loader, dataset = make_earth_dataloaders(
    data_dir="data",
    name="earthquake",
    batch_size=batch_size,
    seed=seed,
    device="cpu",
)
print(f"Dataset size: {len(dataset)}")

# ---- prior score model ----
hidden_dim       = 256
n_hidden_layers  = 4
MODEL_PATH       = "p_score_3_stable.pth"   # adjust path if needed

p_model = AmbientGeneratorScoreNet(hidden_dim=hidden_dim, n_hidden_layers=n_hidden_layers)
p_model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu", weights_only=False))
p_model.eval().to(device)
p_score = DPnPScoreWrapper(p_model)
print("Prior score model loaded.")

## 2. DPnP time schedule and seismic model parameters

In [ ]:
def aneal_schedule(K=20, K0=5, eta0=0.45, etaK=0.15):
    etas = np.zeros(K)
    for i in range(K):
        if i < K0:
            etas[i] = eta0
        else:
            frac = (i - K0) / (K - K0)
            etas[i] = eta0 * (etaK / eta0) ** frac
    return etas

eta = aneal_schedule(K=20, K0=5, eta0=0.1, etaK=0.05)
print("DPnP time schedule (eta):", np.round(eta, 4))

# ---- seismic model parameters ----
BETA   = 10.0     # signal decay rate; higher beta = sharper peak
SIGMA2 = 0.05     # observation noise variance

print(f"\nSeismic model: beta={BETA}, sigma2={SIGMA2}")
print(f"  I(x,y)=1 when x=y; I(x,y)=exp(-2*beta)~{np.exp(-2*BETA):.4f} when antipodal")

## 3. Choose x_true and sensor locations y1, y2

You can either:
- **Sample x_true** from the earthquake test set (cell below), or
- **Specify x_true** directly as a lat/lon pair.

In [ ]:
# ---- Option A: sample x_true from earthquake test set ----
from utils import sample_xtrue_y_batches_from_dataloader

x_true_candidates, _ = sample_xtrue_y_batches_from_dataloader(
    dataloader=test_loader,
    sigma_y=1.0,   # irrelevant here – we only keep x_true
    num_pairs=10,
    device=device,
    dtype=dtype,
    seed=42,
)
# Pick the first candidate
x_true = x_true_candidates[0]   # (3,)

ll = extrinsic_to_latlon_deg_torch(x_true[None]).squeeze()
print(f"x_true (lat, lon) = ({ll[0].item():.2f}°, {ll[1].item():.2f}°)")

# ---- Option B: specify directly ----
# Uncomment and set lat/lon in degrees:
# import math
# lat_deg, lon_deg = 35.0, 139.0   # near Tokyo
# lat_rad = math.radians(lat_deg); lon_rad = math.radians(lon_deg)
# x_true = normalize_torch(torch.tensor(
#     [math.cos(lat_rad)*math.cos(lon_rad),
#      math.cos(lat_rad)*math.sin(lon_rad),
#      math.sin(lat_rad)], dtype=dtype, device=device
# ))

In [ ]:
# ---- Sensor locations ----
# Option A: random sensors on the sphere
torch.manual_seed(7)
y1 = normalize_torch(torch.randn(3, dtype=dtype, device=device))
y2 = normalize_torch(torch.randn(3, dtype=dtype, device=device))

# Option B: specify as lat/lon
# import math
# def latlon_to_s2(lat_deg, lon_deg):
#     lat = math.radians(lat_deg); lon = math.radians(lon_deg)
#     return normalize_torch(torch.tensor(
#         [math.cos(lat)*math.cos(lon), math.cos(lat)*math.sin(lon), math.sin(lat)],
#         dtype=dtype, device=device))
# y1 = latlon_to_s2(0.0,  90.0)   # equator, 90°E
# y2 = latlon_to_s2(0.0, -90.0)   # equator, 90°W

ll1 = extrinsic_to_latlon_deg_torch(y1[None]).squeeze()
ll2 = extrinsic_to_latlon_deg_torch(y2[None]).squeeze()
print(f"Sensor y1 (lat, lon) = ({ll1[0].item():.2f}°, {ll1[1].item():.2f}°)")
print(f"Sensor y2 (lat, lon) = ({ll2[0].item():.2f}°, {ll2[1].item():.2f}°)")

# True signal values
I1_true = seismic_signal(x_true, y1, BETA).item()
I2_true = seismic_signal(x_true, y2, BETA).item()
print(f"\nTrue I(x, y1) = {I1_true:.4f}")
print(f"True I(x, y2) = {I2_true:.4f}")

## 4. Single-trial DPnP reconstruction

Sample one pair $(u_1, u_2)$ and inspect the DPnP output.

In [ ]:
from DPnP import dPnP_sampler_torch_batched
from utils import sphere_mean_torch
import math

sigma = math.sqrt(SIGMA2)

# Sample a single observation pair
torch.manual_seed(0)
u1_single = I1_true + sigma * torch.randn(1).item()
u2_single = I2_true + sigma * torch.randn(1).item()
print(f"Single trial:  u1 = {u1_single:.4f}  (true I1 = {I1_true:.4f})")
print(f"               u2 = {u2_single:.4f}  (true I2 = {I2_true:.4f})")

# Build q_score for this single observation
f_fn_single = get_two_point_seismic_f_fn(y1, y2, u1_single, u2_single, BETA, SIGMA2)
q_score_single = get_bel(
    f_fn=f_fn_single,
    n_paths=5000,
    n_steps=5,
    device=device,
    dtype=dtype,
)

OUT_SAMPLES = 32

X_steps = dPnP_sampler_torch_batched(
    q_score=q_score_single,
    p_score=p_score,
    y=y1[None],           # (1, 3) dummy to set batch B = 1
    out_samples=OUT_SAMPLES,
    eta=eta,
    grw_steps=5,
    seed=0,
    end_only=False,
    device=device,
    dtype=dtype,
)  # (S+1, 1, P, 3)

X_final_single = X_steps[-1, 0]   # (P, 3)
recon_mean_single = sphere_mean_torch(X_final_single, dim=0)  # (3,)

cos_single = (recon_mean_single * x_true).sum().item()
print(f"\nSingle-trial reconstruction cosine = {cos_single:.4f}")

In [ ]:
# Visualise the single-trial result
plot_spherical_kde_mollweide(
    X_final=X_final_single[None],   # expects (B, P, 3) or (P, 3)
    x_true=x_true,
    y_obs=torch.stack([y1, y2]),     # show both sensors
    overlay_samples=True,
    overlay_y=True,
    title=f"Single-trial DPnP | u1={u1_single:.3f}, u2={u2_single:.3f}\n"
          f"cos(recon, x_true)={cos_single:.4f}",
)

## 5. Averaged reconstruction across multiple $(u_1, u_2)$ samples

`run_two_point_earthquake_demo` batches all K trials into one DPnP call and
produces:
1. A Mollweide KDE of the per-trial averaged reconstructions.
2. A cosine-similarity-vs-DPnP-step plot for different particle counts N.

In [ ]:
demo_results = run_two_point_earthquake_demo(
    x_true=x_true,
    y1=y1,
    y2=y2,
    beta=BETA,
    sigma2=SIGMA2,
    p_score=p_score,
    eta=eta,
    n_trials=20,             # K independent (u1, u2) realisations
    n_bel_paths=5000,
    n_bel_steps=5,
    out_samples=32,
    grw_steps=5,
    particle_counts=[1, 5, 10, 20],
    kde_kappa=25.0,
    device=device,
    dtype=dtype,
    seed=0,
)

In [ ]:
# Inspect summary statistics
print("Per-trial cosine similarities:")
cos_vals = demo_results["cos_x_trials"].numpy()
print(f"  mean  = {cos_vals.mean():.4f}")
print(f"  std   = {cos_vals.std():.4f}")
print(f"  min   = {cos_vals.min():.4f}")
print(f"  max   = {cos_vals.max():.4f}")

overall_cos = (demo_results["overall_mean"] * demo_results["x_true"]).sum().item()
print(f"\nOverall averaged reconstruction cosine = {overall_cos:.4f}")

## 6. Cosine similarity vs DPnP steps (multiple x_true from data)

`cosine_similarity_vs_steps_two_point` is the two-point analog of
`cosine_similarity_vs_steps_for_particle_counts`: it samples many `x_true`
values from the earthquake test set, draws one $(u_1,u_2)$ per `x_true`, runs
DPnP, and plots reconstruction quality vs DPnP step.

In [ ]:
summary = cosine_similarity_vs_steps_two_point(
    dataloader=test_loader,
    p_score=p_score,
    eta=eta,
    y1=y1,
    y2=y2,
    beta=BETA,
    sigma2=SIGMA2,
    n_bel_paths=5000,
    n_bel_steps=5,
    particle_counts=[1, 5, 10, 20],
    out_samples=32,
    grw_steps=5,
    num_pairs=40,
    batch_eval_size=8,
    device=device,
    dtype=dtype,
    seed=0,
    show_stderr=True,
)

## 7. Sensitivity: vary the number of observations (n_trials)

Observe how averaging over more $(u_1,u_2)$ realisations improves the
overall reconstruction.

In [ ]:
cos_vs_trials = {}

for n_tr in [1, 5, 10, 30]:
    res = run_two_point_earthquake_demo(
        x_true=x_true,
        y1=y1,
        y2=y2,
        beta=BETA,
        sigma2=SIGMA2,
        p_score=p_score,
        eta=eta,
        n_trials=n_tr,
        n_bel_paths=5000,
        n_bel_steps=5,
        out_samples=32,
        grw_steps=5,
        particle_counts=[10],
        device=device,
        dtype=dtype,
        seed=0,
    )
    overall_cos = (res["overall_mean"] * res["x_true"]).sum().item()
    cos_vs_trials[n_tr] = overall_cos
    print(f"n_trials={n_tr:>3d}  |  overall mean cosine = {overall_cos:.4f}")

In [ ]:
plt.figure(figsize=(6, 4))
ns = list(cos_vs_trials.keys())
cs = [cos_vs_trials[n] for n in ns]
plt.plot(ns, cs, marker="o", linewidth=2)
plt.xlabel("n_trials (number of (u1,u2) realisations)")
plt.ylabel("overall mean cosine similarity")
plt.title("Reconstruction quality vs number of averaged trials")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. (Optional) Explore the single-sensor version

`get_seismic_f_fn` implements the likelihood for a single sensor.  
You can compare reconstruction quality with one vs two sensors.

In [ ]:
# --- single-sensor reconstruction (for comparison) ---
f_fn_1sensor = get_seismic_f_fn(y1, u1_single, BETA, SIGMA2)
q_score_1sensor = get_bel(
    f_fn=f_fn_1sensor,
    n_paths=5000,
    n_steps=5,
    device=device,
    dtype=dtype,
)

X_1s = dPnP_sampler_torch_batched(
    q_score=q_score_1sensor,
    p_score=p_score,
    y=y1[None],
    out_samples=OUT_SAMPLES,
    eta=eta,
    grw_steps=5,
    seed=0,
    end_only=True,
    device=device,
    dtype=dtype,
)  # (1, P, 3)

recon_1s = sphere_mean_torch(X_1s[0], dim=0)
cos_1s = (recon_1s * x_true).sum().item()
print(f"Single-sensor cos = {cos_1s:.4f}  |  Two-sensor cos = {cos_single:.4f}")

plot_spherical_kde_mollweide(
    X_final=X_1s,
    x_true=x_true,
    y_obs=y1[None],
    overlay_samples=True,
    overlay_y=True,
    title=f"Single-sensor DPnP (y1 only) | cos={cos_1s:.4f}",
)